# 第27章 柱状图（bar / barh）

使用柱形长度比较类别大小，并掌握排序、分组、堆积和水平布局。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

比较离散类别的数量、金额、均值或组成。

## 数据结构

一列类别和一列指标；分组或堆积图还需要第二个类别维度。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 width 参数从 0.36 改为 0.6，观察柱形间距变化
2. 移除 bar_label 参数，对比有无数值标签的可读性差异
3. 修改 barh 为 bar，将水平柱状图改为垂直布局


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


transactions = pd.read_csv(f"{base_url}/datasets/uci_online_retail_200k.csv", parse_dates=["InvoiceDate"])
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]
transactions["month"] = transactions["InvoiceDate"].dt.to_period("M").astype("string")
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
monthly_summary = completed.groupby("month").agg(sales=("amount", "sum"), orders=("InvoiceNo", "nunique"))
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18
top_countries = completed.groupby("Country")["amount"].sum().nlargest(4).index
country_rows = transactions[transactions["Country"].isin(top_countries)].copy()
country_rows["flow"] = np.where(country_rows["Quantity"] > 0, "销售", "退货")
country_rows["amount_abs"] = country_rows["amount"].abs()
regional_summary = country_rows.pivot_table(index="Country", columns="flow", values="amount_abs", aggfunc="sum", fill_value=0) / 10_000
regions = regional_summary.index.to_numpy()
online = regional_summary.get("销售", pd.Series(0, index=regional_summary.index)).to_numpy()
offline = regional_summary.get("退货", pd.Series(0, index=regional_summary.index)).to_numpy()
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"UCI Online Retail：{len(transactions):,} 行；图表使用聚合结果与固定样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
totals = online + offline
order = np.argsort(totals)
fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.barh(regions[order], totals[order], color="#1a73e8")
ax.bar_label(bars, padding=4)
ax.set(title="各区域总销售额", xlabel="销售额（万元）")
ax.spines[["top", "right", "left"]].set_visible(False)
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
x = np.arange(len(regions))
width = 0.36
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(x - width / 2, online, width, label="线上", color="#1a73e8")
ax.bar(x + width / 2, offline, width, label="线下", color="#f9ab00")
ax.set(title="区域渠道对比", ylabel="销售额（万元）", xticks=x, xticklabels=regions)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
plt.show()


## 3. 参数说明

- width：柱宽
- bottom：堆积基线
- barh：水平布局
- bar_label：数值标签


## 4. 结果解读

比较共享零基线上的柱长；堆积图同时读取总量和构成，但中间序列不易精确比较。


## 常见误区

- 数值轴不从零开始夸大差异
- 类别太多仍使用竖向柱状图
- 用不同颜色装饰同一序列


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(regions, online, label="线上", color="#1a73e8")
ax.bar(regions, offline, bottom=online, label="线下", color="#f9ab00")
ax.set(title="区域销售渠道构成", ylabel="销售额（万元）")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 本章小结

使用柱形长度比较类别大小，并掌握排序、分组、堆积和水平布局。


### 你已经掌握

- 判断柱状图（bar / barh）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 比较离散类别的数量、金额、均值或组成。 |
| 数据结构 | 一列类别和一列指标；分组或堆积图还需要第二个类别维度。 |
| 结果解读 | 比较共享零基线上的柱长；堆积图同时读取总量和构成，但中间序列不易精确比较。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `width` | 柱宽 |
| `bottom` | 堆积基线 |
| `barh` | 水平布局 |
| `bar_label` | 数值标签 |


### 需要注意

- 数值轴不从零开始夸大差异
- 类别太多仍使用竖向柱状图
- 用不同颜色装饰同一序列


### 完成检查

- [ ] 能判断什么问题适合使用柱状图（bar / barh）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
